# 图像质量分类模型应用 (blur / dark / good / noise)

In [2]:
# ── 0. 配置区 ──────────────────────────────────────────────

from pathlib import Path
import pandas as pd

# FiftyOne 数据集
DATASET_NAME = "TEST2_air2_0701_0823"          # FiftyOne dataset 名称，Master 填写

# 模型路径
MODEL_PATH = "/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/output/swd_cls_a02_image_quaility_DL_v1_noAug_seed42/data_split_0.6_0.2_0.2_yolo11n-cls.pt_4/weights/best.pt"

# FiftyOne 字段名（fo.Classification）
CLF_FIELD  = "pred_quality"

# 推理配置
CLF_DEVICE         = "cuda"
SKIP_IF_CLF_EXISTS = True      # True：已分类的样本跳过

# 输出路径
OUTPUT_PATH = "./exports"

In [3]:
# ── 1. 初始化日志 ──────────────────────────────────────────

import logging

try:
    import ipynbname
    _nb_name = ipynbname.name()
except Exception:
    _nb_name = "a02_1_apply_cls_model"

log_file_name = f"{OUTPUT_PATH}/{_nb_name}_{DATASET_NAME}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ Notebook logging 初始化完成 ============")
logger.info(f"Dataset: {DATASET_NAME} | Model: {MODEL_PATH} | Field: {CLF_FIELD}")

2026-02-27 12:00:20,976 [INFO] ============ Notebook logging 初始化完成 ============
2026-02-27 12:00:20,977 [INFO] Dataset: TEST2_air2_0701_0823 | Model: /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/output/swd_cls_a02_image_quaility_DL_v1_noAug_seed42/data_split_0.6_0.2_0.2_yolo11n-cls.pt_4/weights/best.pt | Field: pred_quality


In [3]:
# ── Step 1：加载 FiftyOne Dataset ─────────────────────────
# 输入：DATASET_NAME（配置区）
# 输出：ds 对象

logger.info("Step 1 开始：加载 FiftyOne Dataset")

import fiftyone as fo
from IPython.display import display

try:
    ds = fo.load_dataset(DATASET_NAME)
    logger.info(f"数据集加载成功：{ds.name}，共 {len(ds)} 张")
except Exception as e:
    logger.error(f"数据集加载失败 {DATASET_NAME}: {e}")
    raise

display(ds)
logger.info("Step 1 完成：FiftyOne Dataset 加载成功")

2026-02-23 22:43:13,026 [INFO] Step 1 开始：加载 FiftyOne Dataset
2026-02-23 22:43:14,043 [INFO] 数据集加载成功：TEST2_ms2_0605_0923，共 12254 张


Name:        TEST2_ms2_0605_0923
Media type:  image
Num samples: 12254
Persistent:  True
Tags:        []
Sample fields:
    id:                                                              fiftyone.core.fields.ObjectIdField
    filepath:                                                        fiftyone.core.fields.StringField
    tags:                                                            fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                                                        fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:                                                      fiftyone.core.fields.DateTimeField
    last_modified_at:                                                fiftyone.core.fields.DateTimeField
    pred_yolo11l_20pct_null_images_add_rawData_batch_4_final:        fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    focus:                           

2026-02-23 22:43:14,045 [INFO] Step 1 完成：FiftyOne Dataset 加载成功


In [ ]:
# ── Step 2：加载 YOLO 分类模型 ─────────────────────────────
# 输入：MODEL_PATH（配置区）
# 输出：clf_model 对象

logger.info("Step 2 开始：加载 YOLO 分类模型")

from ultralytics import YOLO

try:
    clf_model = YOLO(MODEL_PATH)
    logger.info(f"模型加载完成：{MODEL_PATH}")
    logger.info(f"类别：{clf_model.names}")
except Exception as e:
    logger.error(f"模型加载失败 {MODEL_PATH}: {e}")
    raise

logger.info("Step 2 完成：YOLO 分类模型加载成功")

2026-02-23 22:43:20,754 [INFO] Step 2 开始：加载 YOLO 分类模型
2026-02-23 22:43:22,372 [INFO] 模型加载完成：best.pt
2026-02-23 22:43:22,373 [INFO] 类别：{0: 'blur', 1: 'dark', 2: 'good', 3: 'noise'}
2026-02-23 22:43:22,374 [INFO] Step 2 完成：YOLO 分类模型加载成功


In [5]:
# ── Step 3 辅助：推理函数定义 ──────────────────────────────

def predict_quality(image_path: str) -> tuple[str, float]:
    """对单张图推理，返回 (label, confidence)。"""
    results = clf_model.predict(source=image_path, device=CLF_DEVICE, verbose=False)
    r = results[0]
    label = r.names[int(r.probs.top1)]
    score = float(r.probs.top1conf)
    return label, score

In [6]:
# ── Step 3：批量推理，将分类结果写入 FiftyOne ─────────────
# 输入：ds（FiftyOne Dataset），clf_model
# 输出：ds[CLF_FIELD]（fo.Classification）+ sample.tags

logger.info("Step 3 开始：批量推理写入 FiftyOne")

# 字段不存在则创建
if CLF_FIELD not in ds.get_field_schema():
    ds.add_sample_field(
        CLF_FIELD,
        fo.EmbeddedDocumentField,
        embedded_doc_type=fo.Classification,
    )
    logger.info(f"已创建字段：{CLF_FIELD}")

skip_count  = 0
infer_count = 0
error_count = 0

for sample in ds.iter_samples(progress=True, autosave=True):
    if SKIP_IF_CLF_EXISTS and sample[CLF_FIELD] is not None:
        skip_count += 1
        continue
    try:
        label, score = predict_quality(sample.filepath)
        sample[CLF_FIELD] = fo.Classification(label=label, confidence=score)
        if label not in sample.tags:
            sample.tags = sample.tags + [label]
        infer_count += 1
    except Exception as e:
        logger.error(f"推理失败 {sample.filepath}: {e}")
        error_count += 1

logger.info(
    f"Step 3 完成：推理={infer_count} 张，跳过={skip_count} 张，失败={error_count} 张"
)

2026-02-23 22:43:35,615 [INFO] Step 3 开始：批量推理写入 FiftyOne
2026-02-23 22:43:35,629 [INFO] 已创建字段：pred_quality


 100% |█████████████| 12254/12254 [32.7m elapsed, 0s remaining, 6.5 samples/s]      


2026-02-23 23:16:14,948 [INFO]  100% |█████████████| 12254/12254 [32.7m elapsed, 0s remaining, 6.5 samples/s]      
2026-02-23 23:16:14,961 [INFO] Step 3 完成：推理=12254 张，跳过=0 张，失败=0 张


In [ ]:
# ── Step 4：导出统计 Excel ────────────────────────────────
# 输入：ds[CLF_FIELD]（FiftyOne 字段）
# 输出：EXPORT_OUT_DIR 下两个 Excel 文件

logger.info("Step 4 开始：导出统计 Excel")

import pandas as pd

rows = []
for sample in ds.iter_samples(progress=True):
    clf_result = sample[CLF_FIELD]
    if clf_result is None:
        rows.append({"filepath": sample.filepath, "label": None, "confidence": None})
    else:
        rows.append({
            "filepath":   sample.filepath,
            "focus":    sample.focus,
            "datetime": sample.datetime,
            "label":      clf_result.label,
            "confidence": clf_result.confidence,
        })

per_image_df = pd.DataFrame(rows)

summary_df = (
    per_image_df.groupby("label", dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

summary_path   = f"{OUTPUT_PATH}/{DATASET_NAME}_quality_summary.xlsx"
per_image_path = f"{OUTPUT_PATH}/{DATASET_NAME}_quality_per_image.xlsx"

summary_df.to_excel(summary_path, index=False)
per_image_df.to_excel(per_image_path, index=False)

logger.info(f"Step 4 完成：汇总表 → {summary_path}，逐图表 → {per_image_path}")

2026-02-24 00:11:43,953 [INFO] Step 4 开始：导出统计 Excel


 100% |█████████████| 12254/12254 [13.3s elapsed, 0s remaining, 1.3K samples/s]       


2026-02-24 00:11:57,252 [INFO]  100% |█████████████| 12254/12254 [13.3s elapsed, 0s remaining, 1.3K samples/s]       
2026-02-24 00:11:57,620 [INFO] Step 4 完成：汇总表 → exports/quality_summary__TEST2_ms2_0605_0923.xlsx，逐图表 → exports/quality_per_image__TEST2_ms2_0605_0923.xlsx


In [5]:
# ── 验证：展示分类结果 ────────────────────────────────────
# 输入：per_image_df、summary_df
# 展示：逐图表头部 + 标签分布汇总

from IPython.display import display, HTML

per_image_df = pd.read_excel(f"{OUTPUT_PATH}/{DATASET_NAME}_quality_summary.xlsx")
summary_df   = pd.read_excel(f"{OUTPUT_PATH}/{DATASET_NAME}_quality_per_image.xlsx")

display(HTML("<b>每张图分类结果（前10行）：</b>"))
display(per_image_df)

display(HTML("<b>标签分布汇总：</b>"))
display(summary_df)

logger.info(
    f"验证展示完成：共 {len(per_image_df)} 张，"
    f"标签分布：{summary_df.set_index('label')['count'].to_dict()}"
)

FileNotFoundError: [Errno 2] No such file or directory: './exports/TEST2_air2_0701_0823_quality_summary.xlsx'

In [19]:
# label 改成数字
label_mapping = {
    "good": 4,
    "noise": 3,
    "blur": 2,
    "dark": 1,
    None: -1
}
per_image_df["label"] = per_image_df["label"].map(label_mapping)


In [21]:
import dtale
dtale.show(per_image_df).open_browser()

/home/tianqi/miniconda3/envs/fif/lib/python3.9/site-packages/dtale/charts/utils.py:185: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

/home/tianqi/miniconda3/envs/fif/lib/python3.9/site-packages/dtale/charts/utils.py:185: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

/home/tianqi/miniconda3/envs/fif/lib/python3.9/site-packages/dtale/charts/utils.py:185: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

/home/tianqi/miniconda3/envs/fif/lib/python3.9/site-packages/dtale/charts/utils.py:185: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

/home/tianqi/miniconda3/envs/fif/lib/python3.9/site-packages/dtale/charts/utils.py:185: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

2026-02-24 00:35:36,625 [ERROR] Exception on /dtale/charts/_